In [ ]:
!pip install langgraph langchain google-generativeai langchain-google-genai

In [ ]:

import json
from typing import TypedDict, Dict, Any
from langgraph.graph import StateGraph, END
import google.generativeai as genai

In [ ]:
GEMINI_API_KEY = ""

genai.configure(api_key='API')

model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
def ask_gemini(prompt):

    response = model.generate_content(prompt)

    return response.text


In [ ]:


class TelecomState(TypedDict):
    complaint: str
    issue_type: str
    severity: str
    impact_scope: str
    sentiment: str
    backend_results: Dict[str, Any]
    assigned_agent: str
    resolution: str
    technician_notes: str
    eta: str

In [ ]:

# STAGE 1 - COMPLAINT UNDERSTANDING


def complaint_understanding(state):

    complaint = state["complaint"]

    prompt = f"""
    Analyze this telecom complaint.

    Complaint:
    {complaint}

    Return JSON with:
    {{
      "issue_type":"",
      "severity":"",
      "impact_scope":"",
      "sentiment":""
    }}
    """

    result = ask_gemini(prompt)

    print("Gemini Response:\n", result)

    try:
        data = json.loads(result)

    except:
        data = {
            "issue_type": "Unknown",
            "severity": "Medium",
            "impact_scope": "Individual",
            "sentiment": "Negative"
        }

    state["issue_type"] = data["issue_type"]
    state["severity"] = data["severity"]
    state["impact_scope"] = data["impact_scope"]
    state["sentiment"] = data["sentiment"]

    return state


# STAGE 2 - BACKEND INVESTIGATION


def backend_investigation(state):

    issue = state["issue_type"]

    results = {}

    if "tower" in issue.lower():
        results["tower_health"] = "Tower Down"
        results["outage_api"] = "Major outage detected"

    elif "billing" in issue.lower():
        results["billing_system"] = "Incorrect charges detected"

    elif "sim" in issue.lower():
        results["device_db"] = "SIM registration mismatch"

    else:
        results["crm"] = "No major issue found"

    print("\nBackend Results:", results)

    state["backend_results"] = results

    return state

# STAGE 3 - INTELLIGENT ROUTING


def telecom_router(state):

    issue = state["issue_type"].lower()
    severity = state["severity"].lower()

    if severity == "critical":
        return "escalation_agent"

    elif "billing" in issue:
        return "billing_agent"

    elif "tower" in issue or "outage" in issue:
        return "network_agent"

    elif "sim" in issue:
        return "device_agent"

    else:
        return "priority_agent"









In [ ]:

# AGENTS


def billing_agent(state):

    print("\nRouting -> Billing Agent")

    state["assigned_agent"] = "Billing Team"

    return state

def network_agent(state):

    print("\nRouting -> Network Operations Agent")

    state["assigned_agent"] = "Network Operations Team"

    return state

def device_agent(state):

    print("\nRouting -> Device Support Agent")

    state["assigned_agent"] = "Device Support Team"

    return state

def escalation_agent(state):

    print("\nRouting -> Escalation Agent")

    state["assigned_agent"] = "Critical Escalation Team"

    return state

def priority_agent(state):

    print("\nRouting -> Priority Handling Agent")

    state["assigned_agent"] = "Priority Support Team"

    return state

In [ ]:

# STAGE 4 - RESOLUTION GENERATION


def generate_resolution(state):

    prompt = f"""
    Generate:
    1. Customer response
    2. Technician notes
    3. Estimated resolution time

    Complaint:
    {state['complaint']}

    Issue Type:
    {state['issue_type']}

    Backend Results:
    {state['backend_results']}
    """

    response = ask_gemini(prompt)

    state["resolution"] = response
    state["technician_notes"] = response
    state["eta"] = "Generated by Gemini"

    return state

In [ ]:

# BUILD TELECOM GRAPH


telecom_builder = StateGraph(TelecomState)

telecom_builder.add_node(
    "complaint_understanding",
    complaint_understanding
)

telecom_builder.add_node(
    "backend_investigation",
    backend_investigation
)

telecom_builder.add_node("billing_agent", billing_agent)
telecom_builder.add_node("network_agent", network_agent)
telecom_builder.add_node("device_agent", device_agent)
telecom_builder.add_node("escalation_agent", escalation_agent)
telecom_builder.add_node("priority_agent", priority_agent)

telecom_builder.add_node(
    "generate_resolution",
    generate_resolution
)

telecom_builder.set_entry_point(
    "complaint_understanding"
)

telecom_builder.add_edge(
    "complaint_understanding",
    "backend_investigation"
)

telecom_builder.add_conditional_edges(
    "backend_investigation",
    telecom_router
)

telecom_builder.add_edge(
    "billing_agent",
    "generate_resolution"
)

telecom_builder.add_edge(
    "network_agent",
    "generate_resolution"
)

telecom_builder.add_edge(
    "device_agent",
    "generate_resolution"
)

telecom_builder.add_edge(
    "escalation_agent",
    "generate_resolution"
)

telecom_builder.add_edge(
    "priority_agent",
    "generate_resolution"
)

telecom_builder.add_edge(
    "generate_resolution",
    END
)

telecom_graph = telecom_builder.compile()


In [ ]:

# RUN TELECOM WORKFLOW


telecom_input = {
    "complaint":
    "Entire area has no signal since morning"
}

telecom_result = telecom_graph.invoke(
    telecom_input
)

print("\nFINAL TELECOM RESULT\n")
print(telecom_result)

In [ ]:

# 2. HEALTHCARE DOMAIN


print("\n HEALTHCARE WORKFLOW\n")

class HealthState(TypedDict):
    symptoms: str
    extracted_symptoms: str
    severity: str
    emergency: bool
    specialist: str
    triage_summary: str

In [ ]:

# STAGE 1


def symptom_analysis(state):

    symptoms = state["symptoms"]

    prompt = f"""
    Analyze these medical symptoms:

    {symptoms}

    Return JSON:
    {{
      "extracted_symptoms":"",
      "severity":"",
      "emergency":"true/false"
    }}
    """

    result = ask_gemini(prompt)

    print("Gemini Response:\n", result)

    try:
        data = json.loads(result)

    except:
        data = {
            "extracted_symptoms": symptoms,
            "severity": "Medium",
            "emergency": False
        }

    state["extracted_symptoms"] = data["extracted_symptoms"]
    state["severity"] = data["severity"]
    state["emergency"] = data["emergency"]

    return state


In [ ]:

# STAGE 2


def medical_retrieval(state):

    print("\nRetrieving Medical Guidelines...")

    return state


In [ ]:

# ROUTING


def healthcare_router(state):

    symptoms = state["symptoms"].lower()

    if state["emergency"]:

        return "emergency_agent"

    elif "rash" in symptoms:

        return "pharmacy_agent"

    elif "dizziness" in symptoms:

        return "neuro_agent"

    else:

        return "general_agent"

In [ ]:

# AGENTS


def emergency_agent(state):

    print("\nRouting -> Emergency Agent")

    state["specialist"] = "Emergency Department"

    return state

def pharmacy_agent(state):

    print("\nRouting -> Pharmacy Review Agent")

    state["specialist"] = "Pharmacy"

    return state

def neuro_agent(state):

    print("\nRouting -> Neurology Agent")

    state["specialist"] = "Neurology"

    return state

def general_agent(state):

    print("\nRouting -> General Physician")

    state["specialist"] = "General Medicine"

    return state


In [ ]:

# STAGE 4


def generate_triage_summary(state):

    prompt = f"""
    Create patient triage summary.

    Symptoms:
    {state['symptoms']}

    Severity:
    {state['severity']}

    Specialist:
    {state['specialist']}
    """

    response = ask_gemini(prompt)

    state["triage_summary"] = response

    return state


In [ ]:

# BUILD HEALTHCARE GRAPH


health_builder = StateGraph(HealthState)

health_builder.add_node(
    "symptom_analysis",
    symptom_analysis
)

health_builder.add_node(
    "medical_retrieval",
    medical_retrieval
)

health_builder.add_node(
    "emergency_agent",
    emergency_agent
)

health_builder.add_node(
    "pharmacy_agent",
    pharmacy_agent
)

health_builder.add_node(
    "neuro_agent",
    neuro_agent
)

health_builder.add_node(
    "general_agent",
    general_agent
)

health_builder.add_node(
    "generate_triage_summary",
    generate_triage_summary
)

health_builder.set_entry_point(
    "symptom_analysis"
)

health_builder.add_edge(
    "symptom_analysis",
    "medical_retrieval"
)

health_builder.add_conditional_edges(
    "medical_retrieval",
    healthcare_router
)

health_builder.add_edge(
    "emergency_agent",
    "generate_triage_summary"
)

health_builder.add_edge(
    "pharmacy_agent",
    "generate_triage_summary"
)

health_builder.add_edge(
    "neuro_agent",
    "generate_triage_summary"
)

health_builder.add_edge(
    "general_agent",
    "generate_triage_summary"
)

health_builder.add_edge(
    "generate_triage_summary",
    END
)

health_graph = health_builder.compile()

In [ ]:
health_graph

In [ ]:

# RUN HEALTHCARE WORKFLOW


health_input = {
    "symptoms":
    "Chest pain and shortness of breath"
}

health_result = health_graph.invoke(
    health_input
)

print("\nFINAL HEALTHCARE RESULT\n")
print(health_result)

In [ ]:
print("\n FINANCE WORKFLOW \n")

class FinanceState(TypedDict):
    applicant: Dict[str, Any]
    profile: str
    fraud_risk: str
    debt_ratio: str
    decision: str
    recommendation: str


In [ ]:

# STAGE 1


def applicant_analysis(state):

    applicant = state["applicant"]

    prompt = f"""
    Analyze loan applicant.

    Applicant Data:
    {applicant}

    Return JSON:
    {{
      "profile":"",
      "fraud_risk":"",
      "debt_ratio":""
    }}
    """

    result = ask_gemini(prompt)

    print("Gemini Response:\n", result)

    try:
        data = json.loads(result)

    except:
        data = {
            "profile": "Medium",
            "fraud_risk": "Low",
            "debt_ratio": "40%"
        }

    state["profile"] = data["profile"]
    state["fraud_risk"] = data["fraud_risk"]
    state["debt_ratio"] = data["debt_ratio"]

    return state

In [ ]:

# ROUTING


def finance_router(state):

    fraud = state["fraud_risk"].lower()
    profile = state["profile"].lower()

    try:
        debt = float(
            state["debt_ratio"]
            .replace("%", "")
        )

    except:
        debt = 50

    if fraud == "high":

        return "fraud_agent"

    elif profile == "excellent":

        return "fasttrack_agent"

    elif debt > 70:

        return "rejection_agent"

    else:

        return "manual_review_agent"

In [ ]:


# AGENTS


def fraud_agent(state):

    print("\nRouting -> Fraud Investigation Agent")

    state["decision"] = "Fraud Investigation Required"

    return state

def fasttrack_agent(state):

    print("\nRouting -> Fast Track Approval Agent")

    state["decision"] = "Loan Approved"

    return state

def rejection_agent(state):

    print("\nRouting -> Rejection Recommendation Agent")

    state["decision"] = "Loan Rejected"

    return state

def manual_review_agent(state):

    print("\nRouting -> Manual Review Agent")

    state["decision"] = "Manual Review Required"

    return state


In [ ]:

# STAGE 4


def generate_finance_summary(state):

    prompt = f"""
    Generate final loan assessment summary.

    Profile:
    {state['profile']}

    Fraud Risk:
    {state['fraud_risk']}

    Debt Ratio:
    {state['debt_ratio']}

    Decision:
    {state['decision']}
    """

    response = ask_gemini(prompt)

    state["recommendation"] = response

    return state

In [ ]:

# BUILD FINANCE GRAPH


finance_builder = StateGraph(FinanceState)

finance_builder.add_node(
    "applicant_analysis",
    applicant_analysis
)

finance_builder.add_node(
    "fraud_agent",
    fraud_agent
)

finance_builder.add_node(
    "fasttrack_agent",
    fasttrack_agent
)

finance_builder.add_node(
    "rejection_agent",
    rejection_agent
)

finance_builder.add_node(
    "manual_review_agent",
    manual_review_agent
)

finance_builder.add_node(
    "generate_finance_summary",
    generate_finance_summary
)

finance_builder.set_entry_point(
    "applicant_analysis"
)

finance_builder.add_conditional_edges(
    "applicant_analysis",
    finance_router
)

finance_builder.add_edge(
    "fraud_agent",
    "generate_finance_summary"
)

finance_builder.add_edge(
    "fasttrack_agent",
    "generate_finance_summary"
)

finance_builder.add_edge(
    "rejection_agent",
    "generate_finance_summary"
)

finance_builder.add_edge(
    "manual_review_agent",
    "generate_finance_summary"
)

finance_builder.add_edge(
    "generate_finance_summary",
    END
)

finance_graph = finance_builder.compile()



In [ ]:
finance_graph

In [ ]:

# RUN FINANCE WORKFLOW


finance_input = {
    "applicant": {
        "salary": 100000,
        "loan_amount": 600000,
        "employment_type": "Full Time",
        "credit_score": 780
    }
}

finance_result = finance_graph.invoke(
    finance_input
)